# Experiment 3 — LB-Simulated Validation & Hyperparameter Tuning

Standard KFold randomly splits rows — but the competition's actual
train/test split is structured differently. The test set contains the **same 10 outlets**
but **different items**. A random 80/20 split is too optimistic because some items
appear in both the training portion and the validation portion.

A better approach is to hold out 40% of items per outlet, forcing the model to generalise
to unseen item-outlet combinations — closer to the actual test structure.

**Notes:**
- 20-seed ensemble (vs single seed in previous experiments)
- GBM dropped — RF alone was cleaner and faster to iterate

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

SEEDS = [42, 7, 13, 99, 2024, 123, 456, 17, 31, 37,
         53, 71, 83, 97, 101, 127, 131, 137, 149, 163]
KF = 5

# ── 1. Load ───────────────────────────────────────────────────────────────────
train = pd.read_csv('train_v9rqX0R.csv')
test  = pd.read_csv('test_AbJTz2l.csv')
print(f"Train: {train.shape} | Test: {test.shape}")

# ── 2. Clean ──────────────────────────────────────────────────────────────────
def clean(df, rw=None, rv=None):
    df = df.copy()
    df['Item_Fat_Content'] = df['Item_Fat_Content'].replace(
        {'LF':'Low Fat','low fat':'Low Fat','reg':'Regular'})
    df['Item_Category'] = df['Item_Identifier'].str[:2]
    df.loc[df['Item_Category']=='NC','Item_Fat_Content'] = 'Non-Consumable'
    if rw is None: rw = df.groupby('Item_Identifier')['Item_Weight'].mean()
    df['Item_Weight'] = df.apply(
        lambda r: rw.get(r['Item_Identifier'], np.nan)
        if pd.isna(r['Item_Weight']) else r['Item_Weight'], axis=1)
    df['Item_Weight'].fillna(df['Item_Weight'].median(), inplace=True)
    df['Outlet_Size'] = df.groupby('Outlet_Type')['Outlet_Size'].transform(
        lambda x: x.fillna(x.mode()[0] if not x.mode().empty else 'Medium'))
    df['Item_Visibility'] = df['Item_Visibility'].replace(0, np.nan)
    if rv is None: rv = df.groupby('Item_Identifier')['Item_Visibility'].mean()
    df['Item_Visibility'] = df.apply(
        lambda r: rv.get(r['Item_Identifier'], np.nan)
        if pd.isna(r['Item_Visibility']) else r['Item_Visibility'], axis=1)
    df['Item_Visibility'].fillna(df['Item_Visibility'].mean(), inplace=True)
    df['Outlet_Age']  = 2013 - df['Outlet_Establishment_Year']
    df['Is_Grocery']  = (df['Outlet_Type'] == 'Grocery Store').astype(int)
    df['MRP_Cluster'] = pd.cut(df['Item_MRP'], bins=[0,70,130,200,270],
                                labels=[0,1,2,3]).astype(int)
    df['ItemType_OutletType'] = df['Item_Type'].astype(str)+'_'+df['Outlet_Type'].astype(str)
    return df, rw, rv

tc, rw, rv = clean(train)
sc, _,  _  = clean(test, rw, rv)

om = tc.groupby('Outlet_Identifier')['Item_MRP'].mean()
tc['Outlet_MRP_Mean'] = tc['Outlet_Identifier'].map(om)
sc['Outlet_MRP_Mean'] = sc['Outlet_Identifier'].map(om)

vm = tc.groupby('Item_Identifier')['Item_Visibility'].mean()
tc['Visibility_MeanRatio'] = tc['Item_Visibility'] / (tc['Item_Identifier'].map(vm) + 1e-6)
sc['Visibility_MeanRatio'] = sc['Item_Visibility'] / (sc['Item_Identifier'].map(vm).fillna(vm.mean()) + 1e-6)

# ── 3. Encode ─────────────────────────────────────────────────────────────────
for col in ['Item_Fat_Content','Item_Type','Outlet_Identifier','Outlet_Size',
            'Outlet_Location_Type','Outlet_Type','Item_Category','ItemType_OutletType']:
    le = LabelEncoder().fit(pd.concat([tc[col], sc[col]]).astype(str))
    tc[col] = le.transform(tc[col].astype(str))
    sc[col] = le.transform(sc[col].astype(str))

# ── 4. Features ───────────────────────────────────────────────────────────────
feats = ['Item_MRP','Outlet_Type','Outlet_Identifier','Outlet_Age','Item_Visibility',
         'Item_Weight','Item_Type','Item_Category','Is_Grocery','MRP_Cluster',
         'Outlet_Size','Outlet_Location_Type','Visibility_MeanRatio',
         'Outlet_MRP_Mean','Item_Fat_Content','ItemType_OutletType']

for df in [tc, sc]:
    for f in feats: df[f] = df[f].fillna(0)

X      = tc[feats].values.astype(float)
y      = train['Item_Outlet_Sales'].values
X_test = sc[feats].values.astype(float)
print(f"Features: {len(feats)}")

# ── 5. RF with 20 seeds, leaf=38 ─────────────────────────────────────────────
# leaf=38 tuned via LB-simulated validation (hold out 40% items per outlet)

print(f"\n=== RF (leaf=38, {len(SEEDS)} seeds) ===")
all_oofs  = []
all_tests = []

for seed in SEEDS:
    kf  = KFold(n_splits=KF, shuffle=True, random_state=seed)
    oof = np.zeros(len(X))
    tp  = np.zeros(len(X_test))
    for tr_i, val_i in kf.split(X):
        m = RandomForestRegressor(
            n_estimators=500,
            min_samples_leaf=38,   # <-- key change from v10 (was 50)
            max_features=0.5,
            n_jobs=-1,
            random_state=seed
        )
        m.fit(X[tr_i], y[tr_i])
        oof[val_i] = m.predict(X[val_i])
        tp += m.predict(X_test) / KF

    rmse = np.sqrt(mean_squared_error(y, oof))
    print(f"  seed={seed:4d}  OOF={rmse:.2f}", flush=True)
    all_oofs.append(oof)
    all_tests.append(tp)

    # Checkpoint every 5 seeds
    if len(all_oofs) % 5 == 0:
        tp_so_far  = np.mean(all_tests, axis=0)
        oof_so_far = np.mean(all_oofs,  axis=0)
        sub = test[['Item_Identifier','Outlet_Identifier']].copy()
        sub['Item_Outlet_Sales'] = tp_so_far.clip(min=0)
        sub.to_csv(f'submission/predictions3_{len(all_oofs)}seeds.csv', index=False)
        print(f"  --> Checkpoint {len(all_oofs)} seeds: "
              f"OOF={np.sqrt(mean_squared_error(y, oof_so_far)):.2f}  "
              f"std={tp_so_far.std():.0f}")

# ── 6. Save final ─────────────────────────────────────────────────────────────
oof_final  = np.mean(all_oofs,  axis=0)
test_final = np.mean(all_tests, axis=0)
print(f"\nFinal {len(SEEDS)}-seed OOF: {np.sqrt(mean_squared_error(y, oof_final)):.2f}  "
      f"std={test_final.std():.0f}")

sub = test[['Item_Identifier','Outlet_Identifier']].copy()
sub['Item_Outlet_Sales'] = test_final.clip(min=0)
sub.to_csv('submission/predictions3.csv', index=False)
print(f"  mean={test_final.mean():.0f}  std={test_final.std():.0f}  "
      f"min={test_final.clip(min=0).min():.0f}")

Train: (8523, 12) | Test: (5681, 11)
Features: 16

=== RF (leaf=38, 20 seeds) ===
  seed=  42  OOF=1080.33
  seed=   7  OOF=1078.23
  seed=  13  OOF=1081.38
  seed=  99  OOF=1079.80
  seed=2024  OOF=1079.91
  --> Checkpoint 5 seeds: OOF=1078.61  std=1318
  seed= 123  OOF=1080.06
  seed= 456  OOF=1077.63
  seed=  17  OOF=1078.79
  seed=  31  OOF=1077.03
  seed=  37  OOF=1080.01
  --> Checkpoint 10 seeds: OOF=1077.86  std=1318
  seed=  53  OOF=1081.57
  seed=  71  OOF=1082.10
  seed=  83  OOF=1081.39
  seed=  97  OOF=1077.40
  seed= 101  OOF=1078.39
  --> Checkpoint 15 seeds: OOF=1078.05  std=1318
  seed= 127  OOF=1078.93
  seed= 131  OOF=1079.28
  seed= 137  OOF=1079.50
  seed= 149  OOF=1081.81
  seed= 163  OOF=1079.31
  --> Checkpoint 20 seeds: OOF=1078.07  std=1318

Final 20-seed OOF: 1078.07  std=1318
  mean=2179  std=1318  min=113
